# Notion Data Extractor (Bulk Extraction Version)

이 노트북은 특정 ID를 수동으로 입력하지 않고, **사용자(인테그레이션)에게 공유된 모든 노션 데이터**를 자동으로 탐색하여 마크다운 파일로 추출합니다.

### 🚀 작동 원리
1. **Search API**: 인테그레이션이 접근 권한을 가진 모든 페이지와 데이터베이스 목록을 가져옵니다.
2. **Auto Crawling**: 찾은 모든 데이터베이스를 쿼리하고, 모든 페이지의 본문을 추출합니다.
3. **Structured Save**: `extracted_notion_data/` 폴더 내에 이름별로 정리하여 저장합니다.

In [ ]:
from notion_client import Client
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

NOTION_TOKEN = os.getenv("NOTION_TOKEN")
notion = Client(auth=NOTION_TOKEN, notion_version="2025-09-03")

output_dir = Path("extracted_notion_data")
output_dir.mkdir(exist_ok=True)

## 1. 보조 함수 정의

In [ ]:
def get_all_blocks(page_id):
    blocks = []
    has_more = True
    start_cursor = None
    while has_more:
        response = notion.blocks.children.list(block_id=page_id, start_cursor=start_cursor)
        blocks.extend(response["results"])
        has_more = response["has_more"]
        start_cursor = response["next_cursor"]
    return blocks

def blocks_to_markdown(blocks):
    md = ""
    for block in blocks:
        b_type = block["type"]
        if b_type in ["paragraph", "heading_1", "heading_2", "heading_3", "bulleted_list_item", "numbered_list_item"]:
            rich_text = block[b_type].get("rich_text", [])
            text = "".join([t["plain_text"] for t in rich_text])
            if not text: continue
            
            if b_type == "heading_1": md += f"# {text}\n\n"
            elif b_type == "heading_2": md += f"## {text}\n\n"
            elif b_type == "heading_3": md += f"### {text}\n\n"
            elif b_type == "bulleted_list_item": md += f"- {text}\n"
            else: md += f"{text}\n\n"
    return md

def get_page_title(page_obj):
    """페이지 객체에서 제목을 추출합니다."""
    properties = page_obj.get("properties", {})
    # 일반 페이지는 'title', 데이터베이스 내 페이지는 'Name' 등 컬럼명이 다를 수 있음
    for prop in properties.values():
        if prop["type"] == "title":
            titles = prop.get("title", [])
            if titles:
                return "".join([t["plain_text"] for t in titles])
    return "Untitled"

## 2. 전체 탐색 및 추출 (Search & Crawl)
이 함수는 접근 가능한 모든 항목을 찾아서 저장합니다.

In [ ]:
def crawl_notion_workspace():
    print("🔍 워크스페이스 탐색 시작...")
    results = notion.search().get("results", [])
    print(f"총 {len(results)}개의 항목(페이지/데이터베이스)을 발견했습니다.\n")

    for item in results:
        obj_type = item["object"]
        obj_id = item["id"]
        
        if obj_type == "page":
            title = get_page_title(item)
            print(f"📄 페이지 추출 중: {title}")
            blocks = get_all_blocks(obj_id)
            markdown = blocks_to_markdown(blocks)
            
            safe_title = "".join([c for c in title if c.isalnum() or c in " _-"]).strip()
            file_path = output_dir / f"{safe_title}_{obj_id[:8]}.md"
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(markdown)
                
        elif obj_type == "database":
            # 데이터베이스는 제목이 별도로 있음
            db_title = "".join([t["plain_text"] for t in item.get("title", [])]) or "Untitled Database"
            print(f"🗃️ 데이터베이스 발견: {db_title} (내부 페이지 탐색 중...)")
            
            # 데이터베이스 내의 모든 페이지 쿼리
            db_pages = notion.databases.query(database_id=obj_id).get("results", [])
            for page in db_pages:
                title = get_page_title(page)
                blocks = get_all_blocks(page["id"])
                markdown = blocks_to_markdown(blocks)
                
                db_dir = output_dir / db_title
                db_dir.mkdir(exist_ok=True)
                
                safe_title = "".join([c for c in title if c.isalnum() or c in " _-"]).strip()
                with open(db_dir / f"{safe_title}.md", "w", encoding="utf-8") as f:
                    f.write(markdown)
    
    print("\n✅ 모든 데이터 추출이 완료되었습니다!")

if NOTION_TOKEN and NOTION_TOKEN != "input_your_token_here":
    crawl_notion_workspace()
else:
    print("NOTION_TOKEN이 설정되지 않았습니다.")